# Notebook 02 — Delta Sharing: Zero-Copy Data Hand-off to Fabric

**Role:** Platform Engineer / Data Architect

**What this shows:** How Fabric reads the same Delta table that Databricks wrote to ADLS Gen2 — **no ETL, no data duplication, no copy jobs** — via a OneLake shortcut.

This notebook runs **in Microsoft Fabric** (attached to the `BankingLakehouse`), after the `silver` OneLake shortcut has been created pointing at the ADLS Gen2 `silver` container that Notebook 01 wrote to in Databricks.

> 🗣️ **Talking Point:** Delta Sharing is the critical integration point. Databricks wrote the data once. Fabric reads it in place via OneLake shortcuts. This eliminates the #1 objection: 'we already have Databricks, why would we pay for Fabric storage?'

In [ ]:
# Import PySpark functions needed to read and validate the shared Delta table
from pyspark.sql import functions as F

# Confirm the Delta table written by the Databricks notebook (Notebook 01) is accessible here via the OneLake shortcut
print('Reading Delta table shared from the Databricks pipeline via OneLake shortcut...')

## Step 1 — Read the Shared Delta Table

> 🗣️ **Talking Point:** In production, a Fabric engineer creates a OneLake shortcut pointing to the ADLS Gen2 container where Databricks writes. No data movement, no pipeline, no SLA lag. Fabric sees the data in real-time.

In [ ]:
# Read the Delta table written by the Databricks ingestion pipeline (Notebook 01) through the
# 'silver' OneLake shortcut created in BankingLakehouse -> Files -> New shortcut (ADLS Gen2, container 'silver')
shared_df = spark.read.format('delta').load('Files/silver/db_silver_transactions')

# Validate the data is intact after the zero-copy hand-off
print(f'Records accessible in Fabric: {shared_df.count()}')
print(f'Columns: {shared_df.columns}')
shared_df.show(5)

## Step 2 — Demonstrate Delta Time Travel Across Platforms

> 🗣️ **Talking Point:** Because it's the same Delta file format, Fabric inherits all Delta capabilities — time travel, ACID transactions, schema evolution. You are not losing any Delta features by reading from Fabric.

In [ ]:
# Use Delta time travel to inspect the table history — works identically in Fabric and Databricks
# because they both read the same Delta transaction log via the shortcut path
history_df = spark.sql("DESCRIBE HISTORY delta.`Files/silver/db_silver_transactions`")

# Show the Delta table history to demonstrate shared transaction log access
print('Delta table history (accessible from both Databricks and Fabric):')
history_df.select('version','timestamp','operation','operationParameters').show(5, truncate=False)

## Step 3 — Show OneLake Shortcut Architecture

> 🗣️ **Talking Point:** A Fabric shortcut is a pointer, not a copy. Storage costs stay in ADLS Gen2. Fabric adds the governance, BI, and ML layer without touching the underlying files. This is the key 'better together' integration point.

In [ ]:
# Storage account backing this demo's ADLS Gen2 containers — matches Notebook 01's config
storage_account_name = 'stbkdemowllakwrlunvf6'

# Describe the OneLake shortcut architecture that makes this zero-copy hand-off possible
architecture = [
    ('Source',       'Azure Data Lake Storage Gen2 (written by Databricks)'),
    ('Format',       'Delta Parquet (open format — no vendor lock-in)'),
    ('Shortcut',     f'Fabric OneLake Shortcut → abfss://silver@{storage_account_name}.dfs.core.windows.net/db_silver_transactions'),
    ('Fabric sees',  'Native Delta table — full read access, time travel, ACID'),
    ('Data copies',  'ZERO — pointer only, no duplication'),
    ('Latency',      'Real-time — Fabric reads Databricks writes immediately'),
    ('Cost',         'Storage billed once in ADLS Gen2')
]

# Print the architecture overview to illustrate zero-copy data sharing
print('=== OneLake Shortcut Architecture ===')
for label, value in architecture:
    print(f'  {label:14}: {value}')

print()

# Validate column counts and null rates to confirm data integrity across the hand-off
print('Data quality check after zero-copy hand-off:')
for col_name in ['TransactionID','Amount','Status','AmountBand']:
    null_count = shared_df.filter(F.col(col_name).isNull()).count()
    print(f'  {col_name}: {null_count} nulls')

## Step 4 — Write Fabric-Enriched Layer Back

> 🗣️ **Talking Point:** Fabric can enrich the data further — joining to Power BI semantic models, adding business rules from Excel uploads, or running Fabric ML models — and write a new Gold layer that Databricks can also read back. The integration is bidirectional.

In [ ]:
# Enrich the shared Databricks data with Fabric-side business logic:
# - Add a risk tier classification based on amount and large-transaction flag
# - Add a Fabric processing timestamp for lineage
fabric_enriched = shared_df \
    .withColumn('RiskTier',
        F.when((F.col('IsLargeTransaction') == True) & (F.col('Status') == 'flagged'), 'High')
         .when(F.col('IsLargeTransaction') == True, 'Medium')
         .otherwise('Standard')) \
    .withColumn('_fabric_processed_at', F.current_timestamp())

# Write the Fabric-enriched Gold layer back as a Delta table — readable by Databricks if needed
fabric_enriched.write.format('delta').mode('overwrite').saveAsTable('fabric_gold_transactions')

# Print enrichment summary with RiskTier distribution
print('Fabric-enriched Gold table written: fabric_gold_transactions')
fabric_enriched.groupBy('RiskTier').count().orderBy('RiskTier').show()